# 04. Feature Engineering - Automobile Loan Default Prediction

Group rare categories, derive new features, and select final features, building on `03_data_preprocessing.ipynb`'s output.


## 1. Setup


In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

processed_dir = 'data/processed' if os.path.exists('data/processed') else '../data/processed'

train_df = pd.read_csv(os.path.join(processed_dir, 'train_processed.csv'))
test_df = pd.read_csv(os.path.join(processed_dir, 'test_processed.csv'))

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (95372, 68)
test_df shape: (23844, 68)


## 2. Group Rare Categories

Categories below 1% of training rows get grouped into `"Other"`. Threshold fit on train only, applied to both.


In [2]:
cols_to_group = ['Type_Organization', 'Client_Education', 'Client_Income_Type', 'Client_Occupation']
min_count = len(train_df) * 0.01

for col in cols_to_group:
    value_counts = train_df[col].value_counts()
    rare_categories = value_counts[value_counts < min_count].index.tolist()
    train_df[col] = train_df[col].replace(rare_categories, 'Other')
    test_df[col] = test_df[col].replace(rare_categories, 'Other')
    print(f"{col}: grouped {len(rare_categories)} rare categories -> {train_df[col].nunique()} categories remain")

Type_Organization: grouped 41 rare categories -> 18 categories remain
Client_Education: grouped 1 rare categories -> 6 categories remain
Client_Income_Type: grouped 4 rare categories -> 6 categories remain
Client_Occupation: grouped 7 rare categories -> 13 categories remain


## 3. Encode Grouped Columns

One-hot encode the 4 now-grouped columns, fit on train, align test.


In [3]:
train_df = pd.get_dummies(train_df, columns=cols_to_group, drop_first=True)
test_df = pd.get_dummies(test_df, columns=cols_to_group, drop_first=True)

train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

dummy_cols = [c for c in train_df.columns if train_df[c].dtype == bool]
train_df[dummy_cols] = train_df[dummy_cols].astype(int)
test_df[dummy_cols] = test_df[dummy_cols].astype(int)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (95372, 103)
test_df shape: (23844, 103)


## 4. Derive New Features

Age in years (from `Age_Days`); `Credit_Amount`/`Loan_Annuity` ratio (implied loan term, r=0.77 from EDA); a household-responsibility flag; a reachability count. (`credit_to_income`/`annuity_to_income`/`income_per_member` were added in `03_data_preprocessing.ipynb`, before the `Client_Income` log-transform, since they need genuine dollar-scale income. `score_mean`/`score_min`/`scores_available` were also added there, before imputation, since they need the genuine missingness pattern.)


In [4]:
for d in (train_df, test_df):
    d['Age_Years'] = d['Age_Days'] / 365
    d['Credit_Loan_Ratio'] = d['Credit_Amount'] / d['Loan_Annuity']
    d['has_children'] = (d['Child_Count'] > 0).astype(int)
    d['contact_count'] = d['Homephone_Tag'] + d['Workphone_Working']

new_feature_cols = ['Age_Years', 'Credit_Loan_Ratio', 'has_children', 'contact_count']
print(train_df[new_feature_cols].describe())

          Age_Years  Credit_Loan_Ratio  has_children  contact_count
count  95372.000000       95372.000000  95372.000000   95372.000000
mean      43.860553          21.787266      0.290630       0.483653
std       11.782259           8.658270      0.454056       0.684840
min       21.030137           1.800180      0.000000       0.000000
25%       34.263014          15.480258      0.000000       0.000000
50%       43.064384          20.000000      0.000000       0.000000
75%       53.509589          27.592439      1.000000       1.000000
max       69.043836         143.557089      1.000000       2.000000


C:\Users\amami\AppData\Local\Temp\ipykernel_28812\2659354868.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['Age_Years'] = d['Age_Days'] / 365
C:\Users\amami\AppData\Local\Temp\ipykernel_28812\2659354868.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['Credit_Loan_Ratio'] = d['Credit_Amount'] / d['Loan_Annuity']
C:\Users\amami\AppData\Local\Temp\ipykernel_28812\2659354868.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

## 5. Feature Selection

Drop `Age_Days` (perfectly redundant with `Age_Years`) and `Loan_Annuity` (r=0.77 with `Credit_Amount`, now captured by `Credit_Loan_Ratio`). Then fit a quick Random Forest on train to rank remaining features and drop those with importance below 0.001.


In [5]:
train_df = train_df.drop(columns=['Age_Days', 'Loan_Annuity'])
test_df = test_df.drop(columns=['Age_Days', 'Loan_Annuity'])

print("Shape after dropping redundant features:", train_df.shape)

Shape after dropping redundant features: (95372, 105)


In [6]:
from sklearn.ensemble import RandomForestClassifier

X_train = train_df.drop(columns=['Default'])
y_train = train_df['Default']

rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

low_importance_threshold = 0.001
features_to_drop = importances[importances < low_importance_threshold].index.tolist()

print("Top 10 features by importance:")
print(importances.head(10))
print()
print(f"Dropping {len(features_to_drop)} features below importance {low_importance_threshold}")

train_df = train_df.drop(columns=features_to_drop)
test_df = test_df.drop(columns=features_to_drop)

print("Final shape:", train_df.shape, test_df.shape)

Top 10 features by importance:
score_mean                    0.226739
score_min                     0.158876
Score_Source_3                0.106050
Score_Source_2                0.104294
Employed_Days                 0.042334
Score_Source_1                0.040890
Age_Years                     0.026402
Client_Education_Secondary    0.022761
Credit_Loan_Ratio             0.021963
Phone_Change                  0.018321
dtype: float64

Dropping 58 features below importance 0.001
Final shape: (95372, 47) (23844, 47)


## 6. Feature Scaling

Standardize continuous numeric columns (deferred from `03_data_preprocessing.ipynb` so engineered features get scaled too). Fit on train only.


In [7]:
from sklearn.preprocessing import StandardScaler

numeric_cols_to_scale = [c for c in train_df.select_dtypes(include=[np.number]).columns
                          if c != 'Default' and train_df[c].nunique() > 2]

scaler = StandardScaler()
train_df[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
test_df[numeric_cols_to_scale] = scaler.transform(test_df[numeric_cols_to_scale])

print("Scaled columns:", numeric_cols_to_scale)
print()
print("train mean after scaling (should be ~0):")
print(train_df[numeric_cols_to_scale].mean().round(2))

Scaled columns: ['Client_Income', 'Child_Count', 'Credit_Amount', 'Population_Region_Relative', 'Employed_Days', 'Registration_Days', 'ID_Days', 'car_age', 'Client_Family_Members', 'Cleint_City_Rating', 'Application_Process_Day', 'Application_Process_Hour', 'Score_Source_1', 'Score_Source_2', 'Score_Source_3', 'Social_Circle_Default', 'Phone_Change', 'Credit_Bureau', 'score_mean', 'score_min', 'scores_available', 'credit_to_income', 'annuity_to_income', 'income_per_member', 'Age_Years', 'Credit_Loan_Ratio', 'contact_count']

train mean after scaling (should be ~0):
Client_Income                 0.0
Child_Count                   0.0
Credit_Amount                 0.0
Population_Region_Relative    0.0
Employed_Days                -0.0
Registration_Days             0.0
ID_Days                      -0.0
car_age                      -0.0
Client_Family_Members         0.0
Cleint_City_Rating           -0.0
Application_Process_Day      -0.0
Application_Process_Hour      0.0
Score_Source_1      

## 7. Save Final Feature Set

Save for the modelling notebooks (`05_baseline_models.ipynb` onward).


In [8]:
train_df.to_csv(os.path.join(processed_dir, 'train_final.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test_final.csv'), index=False)

print(f"Saved train_final.csv: {train_df.shape}")
print(f"Saved test_final.csv: {test_df.shape}")

Saved train_final.csv: (95372, 47)
Saved test_final.csv: (23844, 47)


## 8. Summary

| Step | Decision |
|---|---|
| Rare categories | `Type_Organization` (41 grouped), `Client_Education` (1), `Client_Income_Type` (4), `Client_Occupation` (7) -> `"Other"` |
| Encoding | One-hot for the 4 grouped columns |
| New features | `Age_Years`, `Credit_Loan_Ratio`, `has_children`, `contact_count` (income ratios and score summaries added earlier in `03_data_preprocessing.ipynb`, before their source columns were transformed/imputed) |
| Feature selection | Dropped `Age_Days`, `Loan_Annuity` (redundant); dropped 58 features with RF importance < 0.001 |
| Scaling | StandardScaler on 27 continuous columns, fit on train only |
| Output | `train_final.csv` (95,372 x 47), `test_final.csv` (23,844 x 47) in `data/processed/` |

Top features by importance: `score_mean`, `score_min`, `Score_Source_3/2/1`, `Employed_Days` - the combined score summary is now the single strongest predictor, confirming the team plan's reasoning that it is more robust than any individual score.

Next: `05_baseline_models.ipynb` - implement and compare at least 4 classification algorithms.
